In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Performance Management: Una Qiskit Function di Q-CTRL Fire Opal
*Consulta il [riferimento API](https://docs.quantum.ibm.com/api/functions/q-ctrl-performance-management)*

> **Note:** Le Qiskit Functions sono una funzionalità sperimentale disponibile esclusivamente per gli utenti dei piani IBM Quantum&reg; Premium Plan, Flex Plan e On-Prem (tramite IBM Quantum Platform API). Sono in stato di anteprima e soggette a modifiche.


<Accordion>
<AccordionItem title="Versioni dei pacchetti">

Il codice in questa pagina è stato sviluppato usando i seguenti requisiti.
Si raccomanda di usare queste versioni o versioni più recenti.

```
qiskit[all]~=2.3.1
qiskit-ibm-runtime~=0.45.1
```
</AccordionItem>
</Accordion>
## Panoramica
Fire Opal Performance Management permette a chiunque di ottenere risultati significativi dai computer quantistici su larga scala, senza dover essere esperti di hardware quantistico. Quando esegui circuiti con Fire Opal Performance Management, vengono applicate automaticamente tecniche di soppressione degli errori basate sull'IA, consentendo di scalare problemi più complessi con più gate e qubit. Questo approccio riduce il numero di shot necessari per ottenere la risposta corretta, senza overhead aggiuntivo — garantendo un risparmio significativo sia in termini di tempo di calcolo che di costi.

Performance Management sopprime gli errori e aumenta la probabilità di ottenere la risposta corretta su hardware rumoroso. In altre parole, aumenta il rapporto segnale-rumore. L'immagine seguente mostra come la maggiore precisione abilitata da Performance Management possa ridurre la necessità di shot aggiuntivi nel caso di un algoritmo Quantum Fourier Transform a 10 qubit. Con soli 30 shot, Q-CTRL raggiunge la soglia di confidenza del 99%, mentre il valore predefinito (`QiskitRuntime` Sampler, `optimization_level`=3 e `resilience_level`=1, `ibm_sherbrooke`) richiede 170.000 shot. Ottenendo la risposta giusta più velocemente, risparmi un tempo di calcolo considerevole.

![Visualizzazione del runtime migliorato](../docs/images/guides/qctrl-performance-management/achieve_more.svg)

La funzione Performance Management può essere utilizzata con qualsiasi algoritmo, e puoi facilmente usarla al posto delle [primitive Qiskit Runtime](/guides/primitives) standard. Dietro le quinte, più tecniche di soppressione degli errori lavorano insieme per prevenire gli errori in fase di esecuzione. Tutti i metodi della pipeline Fire Opal sono preconfigurati e indipendenti dall'algoritmo, il che significa che ottieni sempre le migliori prestazioni senza alcuna configurazione.

Per accedere a Performance Management, [contatta Q-CTRL](https://form.typeform.com/to/uOAVDnGg?typeform-source=q-ctrl.com).
## Descrizione
Fire Opal Performance Management offre due opzioni di esecuzione simili alle primitive Qiskit Runtime, così puoi facilmente sostituire Q-CTRL Sampler ed Estimator. Il flusso di lavoro generale per utilizzare la funzione Performance Management è:
1. Definisci il tuo circuito (e gli operatori nel caso dell'Estimator).
2. Esegui il circuito.
3. Recupera i risultati.

Per ridurre il rumore hardware, Fire Opal impiega una serie di tecniche di soppressione degli errori basate sull'IA, illustrate nell'immagine seguente. Con Fire Opal, l'intera pipeline è completamente automatizzata senza alcun bisogno di configurazione.

La pipeline di Fire Opal elimina la necessità di overhead aggiuntivo, come un maggiore tempo di esecuzione quantistica o qubit fisici extra. Tieni presente che il tempo di elaborazione classica rimane un fattore (consulta la sezione [Benchmark](#benchmarks) per le stime, dove "Tempo totale" riflette sia l'elaborazione classica che quella quantistica). A differenza della mitigazione degli errori, che richiede overhead sotto forma di campionamento, la soppressione degli errori di Fire Opal opera sia a livello di gate che di impulso per affrontare le varie fonti di rumore e ridurre la probabilità che si verifichi un errore. Prevenendo gli errori, si elimina la necessità di costosi post-processing.

L'immagine seguente illustra i metodi di soppressione degli errori automatizzati da Fire Opal Performance Management.

![Visualizzazione della pipeline di soppressione degli errori](../docs/images/guides/qctrl-performance-management/error_suppression.svg)

La funzione offre due primitive, Sampler ed Estimator, e gli input e output di entrambe estendono le specifiche implementate per le [primitive Qiskit Runtime V2](/guides/primitive-input-output#pubs).
## Benchmark
I risultati pubblicati di [benchmark algoritmici](https://journals.aps.org/prapplied/abstract/10.1103/PhysRevApplied.20.024034) dimostrano un miglioramento significativo delle prestazioni su vari algoritmi, tra cui Bernstein-Vazirani, quantum Fourier transform, ricerca di Grover, quantum approximate optimization algorithm e variational quantum eigensolver. Il resto di questa sezione fornisce ulteriori dettagli sui tipi di algoritmi che puoi eseguire, nonché sulle prestazioni e sui tempi di esecuzione previsti.

I seguenti studi indipendenti dimostrano come Performance Management di Q-CTRL abiliti la ricerca algoritmica su scala record:
- [Parametrized Energy-Efficient Quantum Kernels for Network Service Fault Diagnosis](https://arxiv.org/abs/2405.09724v1) - quantum kernel learning fino a 50 qubit
- [Tensor-based quantum phase difference estimation for large-scale demonstration](https://arxiv.org/abs/2408.04946) - quantum phase estimation fino a 33 qubit
- [Hierarchical Learning for Quantum ML: Novel Training Technique for Large-Scale Variational Quantum Circuits](https://arxiv.org/abs/2311.12929) - caricamento dati quantistico fino a 21 qubit

La tabella seguente fornisce una guida approssimativa su precisione e tempi di esecuzione da precedenti esecuzioni di benchmark su `ibm_fez`. Le prestazioni su altri dispositivi possono variare. Il tempo di utilizzo è basato su un'ipotesi di 10.000 shot per circuito. Il "Numero di qubit" indicato non rappresenta un limite rigido, ma soglie approssimative entro cui ci si aspetta un'accuratezza della soluzione estremamente consistente. Problemi di dimensioni maggiori sono stati risolti con successo, e si incoraggia il test oltre questi limiti.

| Esempio    | Numero di qubit | Accuratezza | Misura dell'accuratezza | Tempo totale (s) | Utilizzo runtime (s) | Primitiva (Modalità) |
| ---------  | ---------------- | -------------------------- | -------- | ---------- | ------------- |------------- |
| Bernstein–Vazirani  |  50Q    | 100%  | Tasso di successo (percentuale di esecuzioni in cui la risposta corretta è la bitstring con il conteggio più alto)     | 10    | 8         | Sampler |
| Quantum Fourier Transform | 30Q              | 100% | Tasso di successo (percentuale di esecuzioni in cui la risposta corretta è la bitstring con il conteggio più alto)      | 10    | 8        | Sampler |
| Quantum Phase Estimation  | 30Q   | 99,9998%  | Accuratezza dell'angolo trovato: `1- abs(real_angle - angle_found)/pi`  | 10  | 8  | Sampler |
| Simulazione quantistica: modello di Ising (15 passi) | 20Q  | 99,775%   |  $A$ (definito di seguito)  |  60 (per passo)  | 15 (per passo)   | Estimator |
| Simulazione quantistica 2: dinamica molecolare (20 punti temporali) | 34Q  |  96,78%  |  $A_{mean}$ (definito di seguito)   | 10 (per punto temporale)    | 6 (per punto temporale)  | Estimator |

Definizione dell'accuratezza della misura di un valore di aspettazione — la metrica $A$ è definita come segue:
$$
A = 1 - \frac{|\epsilon^{ideal} - \epsilon^{meas}|}{\epsilon^{ideal}_{max} - \epsilon^{ideal}_{min}},
$$
dove $ \epsilon^{ideal} $ = valore di aspettazione ideale,  $ \epsilon^{meas} $ = valore di aspettazione misurato, $\epsilon^{ideal}_{max} $ = valore massimo ideale, e $\epsilon^{ideal}_{min}$ = valore minimo ideale. $A_{mean}$ è semplicemente la media del valore di $A$ su più misurazioni.

Questa metrica viene utilizzata perché è invariante rispetto a traslazioni globali e ridimensionamenti nell'intervallo dei valori ottenibili. In altre parole, indipendentemente dal fatto che tu sposti l'intervallo dei possibili valori di aspettazione verso l'alto o verso il basso, o aumenti la dispersione, il valore di $A$ dovrebbe rimanere consistente.
## Inizia
Fire Opal Performance Management utilizza Qiskit v`2.0.0`, che è la versione consigliata. Le versioni supportate sono Qiskit >=v`2.0.0`.
Autenticati usando la tua [chiave API di IBM Quantum Platform](http://quantum.cloud.ibm.com/) e seleziona la Qiskit Function come segue. (Questo snippet presuppone che tu abbia già [salvato il tuo account](/guides/functions#install-qiskit-functions-catalog-client) nel tuo ambiente locale.)

In [1]:
# This cell is hidden from users. It hides the "...You have imported samplomatic..." warning.
import warnings

warnings.filterwarnings("ignore", message=".*You have imported samplomatic*")

In [2]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# verify that you have access to the function
catalog.list()

[QiskitFunction(qunova/hivqe-chemistry),
 QiskitFunction(global-data-quantum/quantum-portfolio-optimizer),
 QiskitFunction(algorithmiq/tem),
 QiskitFunction(qedma/qesem),
 QiskitFunction(multiverse/singularity),
 QiskitFunction(ibm/circuit-function),
 QiskitFunction(q-ctrl/optimization-solver),
 QiskitFunction(colibritd/quick-pde),
 QiskitFunction(q-ctrl/performance-management),
 QiskitFunction(kipu-quantum/iskay-quantum-optimizer)]

In [3]:
# Access Function
perf_mgmt = catalog.load("q-ctrl/performance-management")

<Admonition type="note" title="Does this function support all IBM backends?">
If you want to use a backend that this function does not currently support, [reach out to Q-CTRL](https://form.typeform.com/to/iuujEAEI?typeform-source=q-ctrl.com) to add support.
</Admonition>

## Estimator primitive

### Estimator example

Use Fire Opal Performance Management's Estimator primitive to determine the expectation value of a single circuit-observable pair.

In addition to the `qiskit-ibm-catalog` and `qiskit` packages, you will also use the `numpy` package to run this example. You can install this package by uncommenting the following cell if you are running this example in a notebook using the IPython kernel.

In [4]:
# %pip install numpy

**2. Esegui il circuito**

Esegui il circuito e, facoltativamente, definisci il backend e il numero di shot.

In [5]:
import numpy as np
from qiskit.circuit.library import iqp
from qiskit.quantum_info import random_hermitian, SparsePauliOp

n_qubits = 50

# Generate a random circuit
mat = np.real(random_hermitian(n_qubits, seed=1234))
circuit = iqp(mat)
circuit.measure_all()

# Define observables as a string
observable = SparsePauliOp("Z" * n_qubits)

In [6]:
# Create PUB tuple
estimator_pubs = [(circuit, observable)]

Puoi usare le familiari [API Qiskit Serverless](/guides/serverless) per verificare lo stato del tuo workload Qiskit Function:

In [7]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend_name = service.least_busy().name

In [8]:
# Run the circuit using Estimator
qctrl_estimator_job = perf_mgmt.run(
    primitive="estimator",
    pubs=estimator_pubs,
    backend_name=backend_name,
)

**3. Recupera il risultato**

In [9]:
qctrl_estimator_job.status()

'QUEUED'

I risultati hanno lo stesso formato di un [risultato Estimator](/guides/estimator-input-output):

In [10]:
# Retrieve the counts from the result list
result = qctrl_estimator_job.result()

The results have the same format as an [Estimator result](/docs/guides/estimator-input-output):

In [22]:
import numpy

result_str = str(result)

with numpy.printoptions(threshold=200):
    print(
        f"The result of the submitted job had {len(result)} PUB "
        f"and has a value:\n {result[0]}\n"
    )

print("The associated PubResult of this job has the following DataBins:")
print(f"{result[0].data}\n")

print(f"And this DataBin has attributes: {result[0].data.keys()}")

print("The expectation values measured from this PUB are:")
print(f"{result[0].data.evs}")

The result of the submitted job had 1 PUB
The result of the submitted job had 1 PUB and has a value:
 PubResult(data=DataBin(evs=0.0195, stds=0.9998098569228051), metadata={'precision': None})

The associated PubResult of this job has the following DataBins:
DataBin(evs=0.0195, stds=0.9998098569228051)

And this DataBin has attributes: dict_keys(['evs', 'stds'])
The expectation values measured from this PUB are:
0.0195


## Primitiva Sampler
### Esempio con Sampler
Usa la primitiva Sampler di Fire Opal Performance Management per eseguire un circuito Bernstein–Vazirani. Questo algoritmo, usato per trovare una stringa nascosta dagli output di una funzione a scatola nera, è un comune algoritmo di benchmark perché ha un'unica risposta corretta.
**1. Crea il circuito**

Definisci la risposta corretta all'algoritmo, la bitstring nascosta e il circuito Bernstein–Vazirani. Puoi regolare la larghezza del circuito semplicemente modificando `circuit_width`.

In [12]:
import qiskit

circuit_width = 35
hidden_bitstring = "1" * circuit_width

# Create circuit, reserving one qubit for BV oracle
bv_circuit = qiskit.QuantumCircuit(circuit_width + 1, circuit_width)
bv_circuit.x(circuit_width)
bv_circuit.h(range(circuit_width + 1))
for input_qubit, bit in enumerate(reversed(hidden_bitstring)):
    if bit == "1":
        bv_circuit.cx(input_qubit, circuit_width)
bv_circuit.barrier()
bv_circuit.h(range(circuit_width + 1))
bv_circuit.barrier()
for input_qubit in range(circuit_width):
    bv_circuit.measure(input_qubit, input_qubit)

# Create PUB tuple
sampler_pubs = [(bv_circuit,)]

**2. Esegui il circuito**

Esegui il circuito e, facoltativamente, definisci il backend e il numero di shot.

In [13]:
# Run the circuit using Sampler
qctrl_sampler_job = perf_mgmt.run(
    primitive="sampler",
    pubs=sampler_pubs,
    backend_name=backend_name,
)

Controlla lo [stato](/guides/functions#check-job-status) del tuo workload Qiskit Function o recupera i [risultati](/guides/functions#retrieve-results) come segue:

In [14]:
# Print the ID so you can use it later, if necessary
print(qctrl_sampler_job.job_id)

qctrl_sampler_job.status()

60fe2fa1-a860-43e4-8615-c6ac4180f93b


'QUEUED'

**3. Retrieve the result**

In [15]:
# Retrieve the job results
sampler_result = qctrl_sampler_job.result()

In [16]:
# Get results for the first (and only) PUB
pub_result = sampler_result[0]
counts = pub_result.data.c.get_counts()

print("Counts for the meas output register (limited to 30 results):")
for i, (bitstring, count) in enumerate(counts.items()):
    if i >= 50:
        print(f"  ... ({len(counts) - 30} more items)")
        break
    print(f"  {bitstring}: {count}")

Counts for the meas output register (limited to 30 results):
  11111111111111111111111111111111111: 1661
  11111111111111111111111111110111111: 60
  11111111111111111111111111111101111: 54
  11111111111111111111111111111110111: 54
  11111111111111011111111111111111111: 46
  11111111111111111110111111111111111: 44
  11111111111111111111111101111111111: 42
  11111111111111111111111110111111111: 42
  11111111111111110111111111111111111: 41
  11111111111111111111111111111111101: 39
  11111111111111111111101111111111111: 38
  11111111111111111111110111111111111: 38
  11111111111111111111111111101111111: 37
  11111111111111111111111111111111110: 36
  11111111111110111111111111111111111: 35
  11111111111111111111111111111011111: 32
  11111111111111101111111111111111111: 32
  01111111111111111111111111111111111: 27
  11111111111111111011111111111111111: 23
  11111111101111111111111111111111111: 22
  11111111111111111111111111111111011: 21
  11111111011111111111111111111111111: 20
  00000000000

**3. Plot the top bitstrings**

Plot the bitstring with the highest counts to see if the hidden bitstring was the mode.

In [17]:
import matplotlib.pyplot as plt


def plot_top_bitstrings(counts_dict, hidden_bitstring=None):
    # Sort and take the top 100 bitstrings
    top_100 = sorted(counts_dict.items(), key=lambda x: x[1], reverse=True)[
        :100
    ]
    if not top_100:
        print("No bitstrings found in the input dictionary.")
        return

    # Unzip the bitstrings and their counts
    bitstrings, counts = zip(*top_100)

    # Assign colors: purple if the bitstring matches hidden_bitstring,
    # otherwise gray
    colors = [
        "#680CE9" if bit == hidden_bitstring else "gray" for bit in bitstrings
    ]

    # Create the bar plot
    plt.figure(figsize=(15, 8))
    plt.bar(
        range(len(bitstrings)), counts, tick_label=bitstrings, color=colors
    )

    # Rotate the bitstrings for better readability
    plt.xticks(rotation=90, fontsize=8)
    plt.xlabel("Bitstrings")
    plt.ylabel("Counts")
    plt.title("Top 100 Bitstrings by Counts")

    # Show the plot
    plt.tight_layout()
    plt.show()

The hidden bitstring is highlighted in purple, and it should be the bitstring with the highest number of counts.

In [18]:
plot_top_bitstrings(counts, hidden_bitstring)

<Image src="../docs/images/guides/q-ctrl-performance-management/extracted-outputs/8106d906-0.avif" alt="Output of the previous code cell" />

**3. Visualizza le bitstring principali**

Visualizza la bitstring con il conteggio più alto per verificare se la bitstring nascosta è la moda.